# 05 - Tier 3: checking every number

Tier 2 runs real queries, so the numbers in the *result* are real. But the
model then writes a paragraph about those results, and nothing forces the
sentence to match the rows.

Tier 3 adds a checking step. Three checks:

| Check | Question it asks |
|---|---|
| 1. Support | is this number actually in the log? |
| 2. Recalculate | if it was worked out, do we get the same? |
| 3. Totals | do the parts add up to the total? |

**There is no AI in the checker.** It is a search through a list and some
arithmetic. If we asked Gemma to check Gemma's maths it would make the same
kind of mistake.


## Setup


In [ ]:
import sys
sys.path.append('../src')

import evidenceiq as eiq


## 1. Test the checker on fake data

We don't need the model or the database to test this. We write a fake log
and a fake answer, and see if the checker catches the mistakes.

This is why this part was not blocked waiting on Tier 2.


In [ ]:
fake_log = []

# the query worked fine
eiq.add_to_log(fake_log, 'run_sql',
               "SELECT invoice_month, net_revenue FROM dim_month ...", True,
               [{'invoice_month': '2011-10', 'revenue': 1069368.23},
                {'invoice_month': '2011-11', 'revenue': 1456145.80}])

# but the model's own calculation divided by the NEW value instead of the
# OLD one, so it got 26.56 where the answer is 36.17
eiq.add_to_log(fake_log, 'run_python',
               "result = {'pct': (1456145.80-1069368.23)/1456145.80*100}", True,
               [{'pct': 26.56}])

# a fake answer: one good claim, one bad calculation, one made-up number
fake_answer = {
    'question': 'test',
    'findings': 'November was higher than October.',
    'log': fake_log,
    'claims': [
        {'text': 'November revenue was 1,456,145.80', 'value': 1456145.80,
         'unit': 'GBP', 'calc': 'none', 'inputs': []},
        {'text': 'an increase of 26.56%', 'value': 26.56, 'unit': '%',
         'calc': 'pct_change', 'inputs': [1456145.80, 1069368.23]},
        {'text': 'October revenue was 1,100,000', 'value': 1100000.0,
         'unit': 'GBP', 'calc': 'none', 'inputs': []},
    ]}

eiq.verify(fake_answer)

for c in fake_answer['claims']:
    print('%-12s %s' % (c['status'], c['text']))
    if c['problem']:
        print('             ->', c['problem'])


We should get:

- claim 1 **supported** - 1,456,145.80 is in the log
- claim 2 **flagged** - 26.56 *is* in the log, so check 1 passes. Only
  check 2 catches it, because redoing the maths gives 36.17.
- claim 3 **unsupported** - 1,100,000 is nowhere in the log

Claim 2 is why we need two separate checks. A number can be in the log and
still be the wrong number to state.


### A few more checks

Small tests, run them whenever you change the checker.


In [ ]:
def check(name, condition):
    print(('PASS  ' if condition else 'FAIL  ') + name)


# a failed query is not evidence
log = []
eiq.add_to_log(log, 'run_sql', 'SELECT SUM(profit) FROM sales', False,
               [{'profit': 500.0}], error='no column profit')
a = eiq.verify({'log': log, 'claims': [
    {'text': 'profit was 500', 'value': 500.0, 'unit': 'GBP',
     'calc': 'none', 'inputs': []}]})
check('a failed query counts as no evidence',
      a['claims'][0]['status'] == 'unsupported')

# rounding is allowed (within 1%)
log = []
eiq.add_to_log(log, 'run_sql', 'SELECT ...', True, [{'revenue': 1456145.80}])
a = eiq.verify({'log': log, 'claims': [
    {'text': 'about 1,456,000', 'value': 1456000.0, 'unit': 'GBP',
     'calc': 'none', 'inputs': []}]})
check('rounding to 1,456,000 is still supported',
      a['claims'][0]['status'] == 'supported')

# the model cannot mark its own homework
a = eiq.verify({'log': log, 'claims': [
    {'text': 'revenue was 999', 'value': 999.0, 'unit': 'GBP',
     'calc': 'none', 'inputs': [], 'status': 'supported'}]})
check('a status set by the model is ignored',
      a['claims'][0]['status'] == 'unsupported')

# shares have to add up
a = eiq.verify({'log': log, 'claims': [
    {'text': 'A is 40%', 'value': 40.0, 'unit': '%',
     'calc': 'share', 'inputs': [400.0, 1000.0]},
    {'text': 'B is 30%', 'value': 30.0, 'unit': '%',
     'calc': 'share', 'inputs': [300.0, 1000.0]}]})
check('shares adding to 70% are caught',
      'add up to 70' in (a.get('limitations') or ''))


## 2. What we send back to the model

When a check fails we don't just delete the claim - we tell the model what
was wrong and let it try again.


In [ ]:
print(eiq.write_feedback(fake_answer))


## 3. Tier 3 = Tier 2 + checking + retry

The loop is short. All the thinking is in `verify`.

```python
for attempt in range(max_retries + 1):
    answer = tier2(question, log, feedback)   # the model works here
    answer = verify(answer)                   # plain Python
    feedback = write_feedback(answer)
    if feedback == '':
        break
```


In [ ]:
answer = eiq.tier3('Did November 2011 beat October 2011 on revenue, and by how much?')

eiq.show(answer)


### Same question, Tier 2 vs Tier 3

Same model, same prompt, same tools. The only difference is the checking.


In [ ]:
q = 'Which five products generated the most revenue in 2011?'

t2 = eiq.tier2(q)
t3 = eiq.tier3(q)

print('TIER 2 (no checking)')
print(t2['findings'])
print(eiq.score(t2))
print()
print('TIER 3 (checked)')
eiq.show(t3)


## 4. The impossible question

The best demo we have. Ask all three tiers about profit margin.
There is no cost data, so the only honest answer is "I can't".


In [ ]:
q = 'What was our profit margin in 2011?'

for tier_name, fn in [('TIER 1', eiq.tier1), ('TIER 2', eiq.tier2),
                      ('TIER 3', eiq.tier3)]:
    a = fn(q)
    print('=' * 60)
    print(tier_name, '- refused?', a['insufficient_data'])
    print(a['findings'][:300])
    print()


## What we found

Note down for the report:

- how many claims got hidden
- did a retry actually fix anything, or did it fail again?
- Tier 3 is slower and uses more tokens - by how much?

Being honest about the cost is better than pretending it is free.
